# [[5,1,3]] Logical Error Calculator via U†EU (star-decoder formalism)

Implements Gottesman QECCbook-2024 §14.4.2–14.4.3.

For a physical Pauli `E_fault` on 5 qubits and incoming syndrome `s_in`, computes:
- the **logical Pauli** L ∈ {I, X, Y, Z} (the decoded logical error)
- the **outgoing syndrome** `s_out` (passed to the next exRec)

This is the core quantity for the syndrome-conditioned logical error table: the non-Markovian structure arises because `s_out` of one exRec becomes `s_in` of the next.

## Section 1: Pauli algebra over F₂ (binary symplectic representation)

In [1]:
import numpy as np
from itertools import product

# A single-qubit Pauli (ignoring phase) is represented as (x, z) ∈ F_2².
#   x=0,z=0  →  I
#   x=1,z=0  →  X
#   x=1,z=1  →  Y  (= iXZ, but phase doesn't affect syndromes or cosets)
#   x=0,z=1  →  Z

PAULIS_1Q = {
    'I': (np.array([0]), np.array([0])),
    'X': (np.array([1]), np.array([0])),
    'Y': (np.array([1]), np.array([1])),
    'Z': (np.array([0]), np.array([1])),
}

PAULI_LABEL = {(0,0):'I', (1,0):'X', (1,1):'Y', (0,1):'Z'}
PAULI_TO_XZ = {'I': (0,0), 'X': (1,0), 'Y': (1,1), 'Z': (0,1)}

def pauli_multiply(xv1, zv1, xv2, zv2):
    """Multiply two n-qubit Paulis (ignoring phase). Returns (xv, zv)."""
    return (xv1 ^ xv2), (zv1 ^ zv2)

def pauli_anticommutes(xv1, zv1, xv2, zv2):
    """Returns True iff the two Paulis anticommute."""
    return bool((np.dot(xv1, zv2) + np.dot(zv1, xv2)) % 2)

def pauli_weight(xv, zv):
    """Number of qubits where Pauli acts non-trivially."""
    return int(np.sum((xv | zv)))

def all_n_qubit_paulis(n):
    """Enumerate all 4^n n-qubit Paulis as list of (xvec, zvec) arrays."""
    paulis = []
    for bits in product([0,1], repeat=2*n):
        xv = np.array(bits[0::2], dtype=int)
        zv = np.array(bits[1::2], dtype=int)
        paulis.append((xv, zv))
    return paulis

def pauli_str_to_vecs(pauli_str):
    """
    Convert a 5-character Pauli string (e.g. 'YXIII') to (xvec, zvec).
    Each character must be one of I, X, Y, Z.
    """
    pauli_str = pauli_str.strip().upper()
    assert len(pauli_str) == 5, f"Expected 5-qubit Pauli string, got length {len(pauli_str)}"
    xv = np.array([PAULI_TO_XZ[c][0] for c in pauli_str], dtype=int)
    zv = np.array([PAULI_TO_XZ[c][1] for c in pauli_str], dtype=int)
    return xv, zv

def vecs_to_pauli_str(xv, zv):
    """Convert (xvec, zvec) to a 5-character Pauli string."""
    return ''.join(PAULI_LABEL[(int(xv[i]), int(zv[i]))] for i in range(5))

print("Section 1 loaded.")

Section 1 loaded.


## Section 2: [[5,1,3]] stabilizer code definition

Stabilizer generators (Gottesman Table 3.2):
- g1 = X Z Z X I
- g2 = I X Z Z X
- g3 = X I X Z Z
- g4 = Z X I X Z

Logical operators: X_L = XXXXX, Z_L = ZZZZZ

In [2]:
N = 5  # number of physical qubits

STAB_GENS = [
    (np.array([1,0,0,1,0]), np.array([0,1,1,0,0])),  # g1: XZZXI
    (np.array([0,1,0,0,1]), np.array([0,0,1,1,0])),  # g2: IXZZX
    (np.array([1,0,1,0,0]), np.array([0,0,0,1,1])),  # g3: XIXZZ
    (np.array([0,1,0,1,0]), np.array([1,0,0,0,1])),  # g4: ZXIXZ
]

XLOG = (np.array([1,1,1,1,1]), np.array([0,0,0,0,0]))  # XXXXX
ZLOG = (np.array([0,0,0,0,0]), np.array([1,1,1,1,1]))  # ZZZZZ

print("Section 2 loaded: [[5,1,3]] code defined.")

Section 2 loaded: [[5,1,3]] code defined.


## Section 3–5: Syndrome computation, correction table, logical class

In [3]:
def compute_syndrome(xv, zv):
    """
    Compute the 4-bit syndrome of Pauli (xv, zv) w.r.t. [[5,1,3]] generators.
    Returns integer in 0..15.
    """
    s = 0
    for i, (gx, gz) in enumerate(STAB_GENS):
        bit = (np.dot(xv, gz) + np.dot(zv, gx)) % 2
        s |= (int(bit) << (3 - i))
    return s

def syndrome_to_bits(s):
    """Convert syndrome integer 0..15 to 4-bit array [s0,s1,s2,s3]."""
    return np.array([(s >> (3-i)) & 1 for i in range(4)], dtype=int)

# Verify generators have syndrome 0
for i, (gx, gz) in enumerate(STAB_GENS):
    assert compute_syndrome(gx, gz) == 0

def build_correction_table():
    """Returns dict {syndrome_int: (xvec, zvec)} — minimum-weight correction."""
    best = {}
    for xv, zv in all_n_qubit_paulis(N):
        s = compute_syndrome(xv, zv)
        w = pauli_weight(xv, zv)
        if s not in best or w < best[s][0]:
            best[s] = (w, xv.copy(), zv.copy())
    return {s: (xv, zv) for s, (w, xv, zv) in best.items()}

CORRECTIONS = build_correction_table()

cx, cz = CORRECTIONS[0]
assert pauli_weight(cx, cz) == 0

LOGICAL_NAMES = ['I', 'X', 'Y', 'Z']

def logical_class(xv, zv):
    """
    Determine the logical Pauli class of E = (xv, zv), assuming E ∈ N(S).
    Returns: (idx, (x_logical, z_logical))  where idx ∈ {0=I, 1=X, 2=Y, 3=Z}.
    """
    x_log = int(pauli_anticommutes(xv, zv, *ZLOG))
    z_log = int(pauli_anticommutes(xv, zv, *XLOG))
    label = (x_log, z_log)
    idx = {(0,0):0, (1,0):1, (1,1):2, (0,1):3}[label]
    return idx, label

assert logical_class(*XLOG) == (1, (1,0))
assert logical_class(*ZLOG) == (3, (0,1))
assert logical_class(np.zeros(5,int), np.zeros(5,int)) == (0, (0,0))

print("Sections 3–5 loaded: syndrome, correction table, logical class ready.")

# Print the correction table for reference
print("\nCorrection table (syndrome → canonical correction Pauli):")
print(f"  {'s_in':>5}  {'bits':>8}  {'correction':>12}  weight")
print(f"  {'-'*40}")
for s in range(16):
    qx, qz = CORRECTIONS[s]
    q_label = vecs_to_pauli_str(qx, qz)
    bits = ''.join(str(b) for b in syndrome_to_bits(s))
    w = pauli_weight(qx, qz)
    print(f"  {s:5d}  {bits:>8}  {q_label:>12}  {w}")

Sections 3–5 loaded: syndrome, correction table, logical class ready.

Correction table (syndrome → canonical correction Pauli):
   s_in      bits    correction  weight
  ----------------------------------------
      0      0000         IIIII  0
      1      0001         XIIII  1
      2      0010         IIZII  1
      3      0011         IIIIX  1
      4      0100         IIIIZ  1
      5      0101         IZIII  1
      6      0110         IIIXI  1
      7      0111         IIIIY  1
      8      1000         IXIII  1
      9      1001         IIIZI  1
     10      1010         ZIIII  1
     11      1011         YIIII  1
     12      1100         IIXII  1
     13      1101         IYIII  1
     14      1110         IIYII  1
     15      1111         IIIYI  1


## Section 6: The core U†EU computation

In [4]:
def u_dagger_E_u(xv_E, zv_E, s_in):
    """
    Compute the logical error and output syndrome for physical Pauli E
    acting on a state with incoming syndrome s_in.

    Steps:
      1. Get canonical correction Q_{s_in} for the incoming syndrome.
      2. E_combined = E · Q_{s_in}  (mod phase)
      3. s_out = syndrome(E_combined)
      4. E_residual = Q_{s_out} · E_combined  (apply correction)
      5. Logical class of E_residual → logical error L

    Parameters
    ----------
    xv_E, zv_E : np.ndarray of shape (5,)
    s_in : int (0..15)

    Returns
    -------
    logical_idx : int  (0=I, 1=X, 2=Y, 3=Z)
    s_out : int  (0..15)
    """
    Qx, Qz = CORRECTIONS[s_in]
    xc, zc = pauli_multiply(xv_E, zv_E, Qx, Qz)
    s_out = compute_syndrome(xc, zc)
    Rx, Rz = CORRECTIONS[s_out]
    xr, zr = pauli_multiply(Rx, Rz, xc, zc)
    logical_idx, _ = logical_class(xr, zr)
    return logical_idx, s_out

print("Section 6 loaded: u_dagger_E_u ready.")

Section 6 loaded: u_dagger_E_u ready.


---
## Interactive Cell: Compute logical error for a given E_fault and s_in

**Instructions:**
- Set `E_fault` as a 5-character string using letters `I`, `X`, `Y`, `Z` (one per qubit).
- Set `s_in` as an integer from `0` to `15` (the incoming syndrome).
- Run the cell to see the logical error and output syndrome.

**Examples:**
- `E_fault = 'IIIII'` — no fault (identity on all 5 qubits)
- `E_fault = 'XIIIII'` — X error on qubit 1 only  *(note: must be 5 chars)*
- `E_fault = 'YXIII'` — Y on qubit 1, X on qubit 3

In [13]:
# ============================================================
#  SET YOUR INPUTS HERE
# ============================================================

E_fault = 'ZIIIX'   # 5-character Pauli string: I, X, Y, Z per qubit
s_in    = 0         # incoming syndrome: integer 0..15

# ============================================================
#  COMPUTATION (no need to edit below)
# ============================================================

assert 0 <= s_in <= 15, f"s_in must be in 0..15, got {s_in}"

xv_E, zv_E = pauli_str_to_vecs(E_fault)
L_idx, s_out = u_dagger_E_u(xv_E, zv_E, s_in)

# Also show the intermediate quantities for transparency
Qx, Qz = CORRECTIONS[s_in]
xc, zc = pauli_multiply(xv_E, zv_E, Qx, Qz)
Rx, Rz = CORRECTIONS[s_out]
xr, zr = pauli_multiply(Rx, Rz, xc, zc)

print("=" * 55)
print("  [[5,1,3]] Logical Error via U†EU")
print("=" * 55)
print(f"  E_fault  = {E_fault}  (weight {pauli_weight(xv_E, zv_E)})")
print(f"  s_in     = {s_in}  ({' '.join(str(b) for b in syndrome_to_bits(s_in))})")
print()
print("  Step-by-step:")
print(f"    Q_{{s_in}}     = {vecs_to_pauli_str(Qx, Qz)}  (canonical correction for s_in)")
print(f"    E_combined = E · Q_{{s_in}} = {vecs_to_pauli_str(xc, zc)}")
print(f"    s_out      = syndrome(E_combined) = {s_out}  ({' '.join(str(b) for b in syndrome_to_bits(s_out))})")
print(f"    Q_{{s_out}}    = {vecs_to_pauli_str(Rx, Rz)}  (correction applied by trailing EC)")
print(f"    E_residual = Q_{{s_out}} · E_combined = {vecs_to_pauli_str(xr, zr)}")
print()
print("  Result:")
print(f"    Logical error L = {LOGICAL_NAMES[L_idx]}")
print(f"    Output syndrome s_out = {s_out}")
print("=" * 55)

  [[5,1,3]] Logical Error via U†EU
  E_fault  = ZIIIX  (weight 2)
  s_in     = 0  (0 0 0 0)

  Step-by-step:
    Q_{s_in}     = IIIII  (canonical correction for s_in)
    E_combined = E · Q_{s_in} = ZIIIX
    s_out      = syndrome(E_combined) = 9  (1 0 0 1)
    Q_{s_out}    = IIIZI  (correction applied by trailing EC)
    E_residual = Q_{s_out} · E_combined = ZIIZX

  Result:
    Logical error L = X
    Output syndrome s_out = 9


---
## Sweep: all s_in for a fixed E_fault

Run this cell to see how the logical error changes across all 16 incoming syndromes for the same `E_fault`. This reveals the syndrome-dependent (non-Markovian) structure.

In [14]:
# Uses the same E_fault defined in the interactive cell above
xv_E, zv_E = pauli_str_to_vecs(E_fault)

print(f"Logical error sweep for E_fault = {E_fault}  (weight {pauli_weight(xv_E, zv_E)})")
print(f"{'s_in':>6}  {'s_in bits':>10}  {'canonical Q':>13}  {'L':>4}  {'s_out':>6}")
print("-" * 50)
for s in range(16):
    L_idx, s_out = u_dagger_E_u(xv_E, zv_E, s)
    Qx, Qz = CORRECTIONS[s]
    bits = ' '.join(str(b) for b in syndrome_to_bits(s))
    print(f"  {s:4d}  {bits:>10}  {vecs_to_pauli_str(Qx, Qz):>13}  {LOGICAL_NAMES[L_idx]:>4}  {s_out:6d}")

Logical error sweep for E_fault = ZIIIX  (weight 2)
  s_in   s_in bits    canonical Q     L   s_out
--------------------------------------------------
     0     0 0 0 0          IIIII     X       9
     1     0 0 0 1          XIIII     Y       8
     2     0 0 1 0          IIZII     Z      11
     3     0 0 1 1          IIIIX     I      10
     4     0 1 0 0          IIIIZ     Z      13
     5     0 1 0 1          IZIII     I      12
     6     0 1 1 0          IIIXI     X      15
     7     0 1 1 1          IIIIY     Y      14
     8     1 0 0 0          IXIII     Y       1
     9     1 0 0 1          IIIZI     X       0
    10     1 0 1 0          ZIIII     I       3
    11     1 0 1 1          YIIII     Z       2
    12     1 1 0 0          IIXII     I       5
    13     1 1 0 1          IYIII     Z       4
    14     1 1 1 0          IIYII     Y       7
    15     1 1 1 1          IIIYI     X       6
